# Milestone 2 lab, notebook 2 of 8: Unique on which columns

## Where we are

| Notebook | What you do |
|---|---|
| 1. Empty in which rows | Write a check for empty cells that does not fire on good rows |
| **2. Unique on which columns** (you are here) | Find the receipt and the file that were counted twice |
| 3. A believable value | Catch 1,180 kg of bananas without catching 400 real bottles |
| 4. Rules of the business | Three rules a person must tell you. Each needs a join first |
| 5. Across two tables | Hold the receipts against the tills' own totals. Find the lines a join dropped |
| 6. A normal day | Find the store that sent nothing. Work out what a normal day is |
| 7. The false alarm | A holiday is not an error. See what silence costs |
| 8. The production version | The same checks as a tested program, and what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
from lab_helpers import bonus, keep, money, prepare_week, show_check, sql
from lotus_pipeline.checks import STANDARD_CHECKS

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August. 24,384 sales lines.
The five standard checks ran on every night: 35 of 35 passed.


## Step 1: A Friday that is too good

**Dana Okafor** manages store `S02`. Here are her last five Fridays, as the pipeline reports them.

In [2]:
sql("""
    SELECT sales_date, receipts, lines, net_sales_cents
    FROM output.daily_store_sales
    WHERE store_id = 'S02' AND extract(isodow FROM sales_date) = 5
    ORDER BY sales_date
""")

,sales_date,receipts,lines,net_sales_cents
0,2026-07-31,47,159,102471
1,2026-08-07,46,165,113649
2,2026-08-14,43,141,95154
3,2026-08-21,42,126,79312
4,2026-08-28,47,334,206856


Four Fridays between about $790 and $1,140. Then Friday 28 August: 206856 cents, $2,068.56.

Look closer at that last row. 334 lines, and only 47 receipts. Twice the lines, with a normal number of customers.

If this is wrong, Dana is overpaid. And the check that should stop it is one of the five: "the key is unique".

## Step 2: The check that should catch it

In [3]:
key_is_unique = STANDARD_CHECKS[2]
print(key_is_unique.sql)
print(len(sql(key_is_unique.sql)), "bad rows. The check passes.")


        SELECT line_id, count(*) AS times
        FROM clean.sales_lines
        GROUP BY line_id
        HAVING count(*) > 1
        
0 bad rows. The check passes.


Zero bad rows. Every `line_id` in the table is there once.

## Step 3: Look at the rows

Take a smaller example first. **Marta Lindqvist** manages store `S01`. On Tuesday 25 August, one of her receipts looks like this.

In [4]:
sql("""
    SELECT line_id, receipt_id, line_no, product_name, quantity, amount_cents
    FROM clean.sales_lines
    WHERE receipt_id = 'S01-T2-004649'
    ORDER BY line_no, line_id
""")

,line_id,receipt_id,line_no,product_name,quantity,amount_cents
0,S01-0825-000050,S01-T2-004649,1,Whole milk 2L,2.00,578
1,S01-0825-000129,S01-T2-004649,1,Whole milk 2L,2.00,578
2,S01-0825-000051,S01-T2-004649,2,Sourdough loaf,1.00,525
3,S01-0825-000130,S01-T2-004649,2,Sourdough loaf,1.00,525
4,S01-0825-000052,S01-T2-004649,3,Cheddar 200g,1.00,449
5,S01-0825-000131,S01-T2-004649,3,Cheddar 200g,1.00,449
6,S01-0825-000053,S01-T2-004649,4,Tomatoes,0.64,255
7,S01-0825-000132,S01-T2-004649,4,Tomatoes,0.64,255


Read it pair by pair. Line 1 is milk, twice. Line 2 is bread, twice. The same receipt, the same position on it, the same product, the same money.

Only one column differs: `line_id`.

That is the whole story. `line_id` is not printed on the receipt. The program that writes the store's file **makes a new one every time it writes a line**. So when it writes a receipt twice, the two copies get two different ids.

The check tested a column that can never be the same twice. It cannot fail.

## Step 4: Unique on which columns?

Ask what names one item line in the real world. Not a number a program made up. It is the receipt, and the position on the receipt: `receipt_id` and `line_no` together.

A key made of more than one column is called a **composite key**.

Write the check on that key.

In [5]:
TRUE_KEY = """
    SELECT receipt_id, line_no, store_id, min(sales_date) AS sales_date, count(*) AS times,
           sum(amount_cents) - min(amount_cents) AS counted_too_much_cents
    FROM clean.sales_lines
    GROUP BY receipt_id, line_no, store_id
    HAVING count(*) > 1
"""
yours = sql(TRUE_KEY)
print(len(yours), "item lines are stored more than once")
yours.head(4)

171 item lines are stored more than once


,receipt_id,line_no,store_id,sales_date,times,counted_too_much_cents
0,S02-T1-004657,3,S02,2026-08-28,2,899
1,S02-T1-004668,3,S02,2026-08-28,2,249
2,S02-T2-004585,6,S02,2026-08-28,2,379
3,S02-T1-004659,5,S02,2026-08-28,2,358


171 item lines. Where are they, and what do they cost?

In [6]:
managers = {"S01": "Marta Lindqvist", "S02": "Dana Okafor"}
by_store = yours.groupby(["store_id", "sales_date"]).agg(
    lines_twice=("times", "size"), counted_too_much=("counted_too_much_cents", "sum")
)
for (store, day), row in by_store.iterrows():
    print(
        f"{store} on {day}: {row.lines_twice:>3} lines twice, {money(row.counted_too_much):>9} counted too much."
        f"  {managers[store]} is overpaid {money(bonus(row.counted_too_much))}."
    )

S01 on 2026-08-25:   4 lines twice,    $18.07 counted too much.  Marta Lindqvist is overpaid $0.18.
S02 on 2026-08-28: 167 lines twice, $1,034.28 counted too much.  Dana Okafor is overpaid $10.34.


Two different faults, found by one check.

- **`S01`, Tuesday:** 4 lines. One receipt was written twice. $18.07 too much.
- **`S02`, Friday:** 167 lines. The store's **whole file** arrived twice. $1,034.28 too much, and $10.34 of bonus.

The size tells them apart: a few lines is a receipt, a whole day is a file.

## Step 5: The opposite mistake

You could also pick too **few** columns. Suppose somebody says: a receipt id is unique, check that.

In [7]:
too_few = sql("""
    SELECT receipt_id, count(*) AS lines
    FROM clean.sales_lines
    GROUP BY receipt_id
    HAVING count(*) > 1
""")
print(len(too_few), "receipts would fail")

6407 receipts would fail


6,407 receipts would fail, and nearly all of them are simply receipts with more than one product.

| The check is unique on | Good data | The two faults |
|---|---|---|
| `line_id`: a column that means nothing | Passes | Passes. Silence |
| `receipt_id`: too few columns | Fails on 6,407 good receipts | |
| `receipt_id` + `line_no`: the true key | Passes | Fails on exactly 171 lines |

## Step 6: Hold it against the production check

In [8]:
keep("true_key_unique", yours)

Your query:           171 bad rows
The production check: 171 bad rows  (One item line is stored once)
The same rows.


In [9]:
show_check("true_key_unique")

2. One item line is stored once
   lens:      data
   answers:   Unique?
   protects:  Store managers and the head of retail operations: net sales, so pay
   prepare:   Group by the key that means something: receipt_id + line_no, not line_id
   limit:     Each receipt_id + line_no is there exactly once

        SELECT receipt_id, line_no, store_id, min(sales_date) AS sales_date, count(*) AS times,
               sum(amount_cents) - min(amount_cents) AS counted_too_much_cents
        FROM clean.sales_lines
        GROUP BY receipt_id, line_no, store_id
        HAVING count(*) > 1


A check that one thing is stored once is called a **unique check**. The work is not the `GROUP BY`. It is choosing the columns.

What can this check not see? A line that is **missing**. It only finds what is there too often. A store that sends nothing passes it perfectly. That is notebook 6.

## What you now know

- A unique check is only as good as the columns it is on.
- A key that a program makes up on every export can never repeat. Checking it proves nothing.
- A composite key is a key made of more than one column: here, `receipt_id` + `line_no`.
- Too few columns gives false alarms. The wrong column gives silence.

**Say it in an interview:** "Our uniqueness check was on a line id that the export program generated fresh on every run, so it could never fail. One store's whole file arrived twice and its sales doubled, with the check green. I moved the check to the composite key, receipt id plus line number, and it found 171 duplicated lines."

**The theory:** [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), section "A check is a query". Composite keys are in [SQL: Data Quality, Debugging & Auditing](https://www.d8loop.com/modules/knowledge-corner/sql/data-quality-debugging-auditing).

Next: [notebook 3](03_a_believable_value.ipynb). One line of bananas costs more than a store sells in a day.